In [0]:

import time
from pyspark.sql.functions import col, sum as _sum

ruta_base = "/Volumes/workspace/tfinal/tfinal/RESULTADOS-PanaderiaZerrillo.csv"

copias = 1000        
repeticiones = 3     

formatos = ["CSV", "Parquet", "Delta"]
rutas = {f: f"{ruta_base}/ventas_{f.lower()}" for f in formatos}




In [0]:

# 1. DATOS BASE Y AMPLIACION
df_base = spark.table("ventas_zerrillo_limpias")
filas_base = df_base.count()

df_grande = df_base.crossJoin(
    spark.range(copias).withColumnRenamed("id", "copia")
)
filas_grande = df_grande.count()

print("Filas base:", filas_base)
print("Filas ampliadas:", filas_grande)
print("Columnas:", df_grande.columns)


Filas base: 1767
Filas ampliadas: 1767000
Columnas: ['id_venta', 'fecha', 'producto', 'categoria', 'cantidad', 'precio_unitario', 'importe_total', 'metodo_pago', 'copia']


In [0]:

# 2. ESCRITURA EN CADA FORMATO (tiempo)

def escribir(formato, ruta):
    escritor = df_grande.repartition(8).write.mode("overwrite")
    if formato == "CSV":
        escritor = escritor.option("header", "true")
    t0 = time.time()
    escritor.format(formato.lower()).save(ruta)
    return time.time() - t0


t_escritura = {}
for f in formatos:
    t_escritura[f] = escribir(f, rutas[f])
    print(f"{f}: escritura {t_escritura[f]:.2f} s")



CSV: escritura 3.66 s
Parquet: escritura 2.66 s
Delta: escritura 2.65 s


In [0]:


# 3. TAMANO EN DISCO
def tamano_bytes(ruta):
    total = 0
    for archivo in dbutils.fs.ls(ruta):
        if archivo.name.endswith("/"):
            total += tamano_bytes(archivo.path)
        else:
            total += archivo.size
    return total


tamano_mb = {f: tamano_bytes(rutas[f]) / (1024 * 1024) for f in formatos}
for f in formatos:
    print(f"{f}: {tamano_mb[f]:.2f} MB")



CSV: 119.17 MB
Parquet: 0.33 MB
Delta: 0.70 MB


In [0]:

# 4. LECTURA Y CONSULTAS (promedio de corridas)

from pyspark.sql.functions import col, sum as _sum

# Nombres reales de las columnas en tu tabla
col_importe = "importe_total" if "importe_total" in df_grande.columns else "importe"
col_pago = next(c for c in ["forma_pago", "metodo_pago"] if c in df_grande.columns)
print("Columna de importe:", col_importe, "| columna de pago:", col_pago)

# Esquema del CSV tomado de la propia tabla
esquema_csv = df_grande.schema


def leer(formato, ruta):
    if formato == "CSV":
        return spark.read.schema(esquema_csv).option("header", "true").csv(ruta)
    return spark.read.format(formato.lower()).load(ruta)


def consulta1(df):
    # Agregacion total: importe por categoria
    return df.groupBy("categoria").agg(_sum(col_importe).alias("total")).collect()


def consulta2(df):
    # Consulta selectiva: ventas Yape en un rango de fechas
    return df.filter(
        (col("fecha") >= "2026-09-03") &
        (col("fecha") <= "2026-09-05") &
        (col(col_pago) == "Yape")
    ).count()


def promedio(consulta, formato, ruta):
    tiempos = []
    for _ in range(repeticiones):
        # En compute serverless CLEAR CACHE no esta soportado
        try:
            spark.catalog.clearCache()
        except Exception:
            pass
        df = leer(formato, ruta)
        t0 = time.time()
        consulta(df)
        tiempos.append(time.time() - t0)
    return sum(tiempos) / len(tiempos), tiempos


resultados = []
for f in formatos:
    prom1, det1 = promedio(consulta1, f, rutas[f])
    prom2, det2 = promedio(consulta2, f, rutas[f])
    print(f"{f} | consulta1: {[round(t, 2) for t in det1]} | consulta2: {[round(t, 2) for t in det2]}")
    resultados.append((
        f,
        round(tamano_mb[f], 2),
        round(t_escritura[f], 2),
        round(prom1, 2),
        round(prom2, 2),
    ))


Columna de importe: importe_total | columna de pago: metodo_pago
CSV | consulta1: [1.48, 1.09, 1.1] | consulta2: [1.03, 1.07, 1.02]
Parquet | consulta1: [0.98, 0.82, 0.83] | consulta2: [0.91, 0.86, 0.84]
Delta | consulta1: [0.74, 0.55, 0.52] | consulta2: [0.55, 0.54, 0.63]


In [0]:


# 5. TABLA DE RESULTADOS (tomar captura)
df_resultados = spark.createDataFrame(
    resultados,
    "formato STRING, tamano_mb DOUBLE, escritura_s DOUBLE, "
    "consulta1_agregacion_s DOUBLE, consulta2_filtro_s DOUBLE",
)
display(df_resultados)

formato,tamano_mb,escritura_s,consulta1_agregacion_s,consulta2_filtro_s
CSV,119.17,3.66,1.22,1.04
Parquet,0.33,2.66,0.87,0.87
Delta,0.7,2.65,0.6,0.58
